# Model Development: Bitcoin and Ethereum Forecasting

This notebook implements and evaluates five forecasting models -- Naive (Persistence), ARIMA, Random Forest, XGBoost, and LSTM -- using the feature sets produced in Notebook 06, across three forecast horizons (1, 7, and 14 days ahead). The target variable is the horizon-day-ahead cumulative log return, already defined and shifted in the feature engineering stage and not recomputed here.

Each model is evaluated across four feature configurations, forming a **Feature Contribution Analysis**: Market Only (the baseline), Market + Google Trends, Market + Fear & Greed Index, and Market + Google Trends + Fear & Greed Index. This design isolates the individual and combined contribution of each external signal relative to market data alone.

A chronological 70%/15%/15% train/validation/test split is used, as no fixed split was specified in the methodology chapter. Random Forest and XGBoost are tuned using time-series-respecting validation strategies; LSTM uses early stopping against the validation set. The test set is evaluated exactly once per model.

Comparative evaluation across models and feature configurations is reserved for Notebook 08.

In [1]:
import sys
print(sys.executable)
print(sys.version)
import subprocess
result = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if any(pkg in line.lower() for pkg in ["tensorflow", "keras", "numpy", "scikit-learn", "xgboost"]):
        print(line)

/Users/mas364/Desktop/AI_Crypto_Forecasting/.venv/bin/python
3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 21.0.0 (clang-2100.0.123.102)]
keras==3.15.1
numpy==2.4.6
scikit-learn==1.9.0
tensorflow==2.21.0
xgboost==3.4.0


In [2]:
import sys
import os
import tensorflow as tf

print("PYTHON:", sys.executable)
print("CWD:", os.getcwd())
print("TF:", tf.__version__)
print("DEVICES:", tf.config.list_physical_devices())

x = tf.constant([[1., 2.], [3., 4.]])
print("TF TEST:", tf.reduce_sum(x).numpy())

PYTHON: /Users/mas364/Desktop/AI_Crypto_Forecasting/.venv/bin/python
CWD: /Users/mas364/Desktop/AI_Crypto_Forecasting/notebooks
TF: 2.21.0
DEVICES: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
TF TEST: 10.0


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("sys.path[0]:", sys.path[0])

PROJECT_ROOT: /Users/mas364/Desktop/AI_Crypto_Forecasting
src exists: True
sys.path[0]: /Users/mas364/Desktop/AI_Crypto_Forecasting


In [4]:
from src.models import lstm_model

print("LSTM_EPOCHS:", lstm_model.LSTM_EPOCHS)
print("LSTM_GRID:", len(lstm_model.LSTM_GRID))
print("TF intra:", __import__("tensorflow").config.threading.get_intra_op_parallelism_threads())
print("TF inter:", __import__("tensorflow").config.threading.get_inter_op_parallelism_threads())

LSTM_EPOCHS: 80
LSTM_GRID: 9
TF intra: 0
TF inter: 0


/Users/mas364/Desktop/AI_Crypto_Forecasting/src/models/lstm_model.py:168: RuntimeWarning: Could not set TensorFlow intra/inter-op thread counts -- TensorFlow has already initialized elsewhere in this process (e.g. an earlier cell already used TensorFlow in this kernel session). Training will proceed with TensorFlow's default thread configuration instead. This is a real, measured risk-reduction step for a confirmed intermittent TensorFlow-internal deadlock on some platforms -- not having it active does not mean training will fail, only that this particular safeguard isn't in effect for the rest of this session. Restart the kernel before running this notebook if you want this setting applied. For a hard guarantee against the same underlying issue regardless of thread count, see src/models/lstm_isolated.py.
  _configure_tf_threading()


In [5]:
import sys
sys.path.insert(0, "/Users/mas364/Desktop/AI_Crypto_Forecasting")

from src.models import lstm_model
import tensorflow as tf
print("TF intra:", tf.config.threading.get_intra_op_parallelism_threads())
print("TF inter:", tf.config.threading.get_inter_op_parallelism_threads())

TF intra: 0
TF inter: 0


In [6]:
import sys
sys.path.insert(0, "/Users/mas364/Desktop/AI_Crypto_Forecasting")

import tensorflow as tf
x = tf.constant([[1., 2.], [3., 4.]])
print("TF sanity check:", tf.reduce_sum(x).numpy())

from src.models import lstm_model
print("LSTM_EPOCHS:", lstm_model.LSTM_EPOCHS)
print("LSTM_GRID:", len(lstm_model.LSTM_GRID))


TF sanity check: 10.0
LSTM_EPOCHS: 80
LSTM_GRID: 9


In [7]:
import os, sys, json, time, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("no xgboost - pip install xgboost")

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("no tensorflow - pip install tensorflow")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
if HAS_TF:
    keras.utils.set_random_seed(SEED)

pd.set_option("display.max_columns", None)

In [8]:
# same root-finding trick as notebooks 05/06
def find_project_root(start, marker):
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"can't find {marker} anywhere above {start}")

PROJECT_ROOT = find_project_root(Path.cwd(), "src")
FEATURES_DIR = PROJECT_ROOT / "data/processed/features"

OUT_DIR = PROJECT_ROOT / "results/modeling"
PRED_DIR = OUT_DIR / "predictions"
MODELS_DIR = OUT_DIR / "fitted_models"
METRICS_DIR = OUT_DIR / "metrics"

for d in [PRED_DIR, MODELS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

# Reusable modelling logic (target/predictor split, all five models, and
# evaluation metrics) now lives in src/, imported here rather than
# redefined inline -- the same pattern Notebook 06 already uses for
# feature engineering. This notebook remains the place where every
# hyperparameter grid, intermediate output, table, and diagnostic is
# actually visible and printed; src/ is only the reusable *mechanism*.
from src.models.data_prep import get_X_y
from src.evaluation.metrics import rmse, mae, mape, dir_acc, r_squared, to_price, evaluate
from src.models.baseline import run_naive
from src.models.arima_model import pick_arima_order, arima_rolling, run_arima
from src.models.tree_models import tune_rf, run_random_forest, run_xgboost
from src.models.lstm_model import make_sequences, fit_lstm_once, run_lstm

print(PROJECT_ROOT)

/Users/mas364/Desktop/AI_Crypto_Forecasting


In [9]:
ASSETS = {"Bitcoin": "Bitcoin_Price", "Ethereum": "Ethereum_Price"}

# Forecast horizons compared side by side, per the supervisor's request
# (1, 7, and 14 days ahead), rather than switching to a single new one.
# Loads the "_7day"/"_14day" feature files produced by the updated
# Notebook 06 -- separate files from the 1-day originals, never
# overwriting them.
FORECAST_HORIZONS = [1, 7, 14]

def horizon_suffix_for(horizon):
    return f"_{horizon}day" if horizon != 1 else ""

FEATURE_FILE_TEMPLATES = {
    "primary": "{a}_market_only_features{s}.csv",
    "candidate": "{a}_market_google_trends_features{s}.csv",
    "ablation": "{a}_market_fear_greed_features{s}.csv",
    "combined": "{a}_market_all_external_features{s}.csv",
}
FEATURE_FILES = FEATURE_FILE_TEMPLATES  # kept for backward-compatible reference elsewhere

# FEATURE_FILES keys above are retained as internal file-lookup keys only
# (they match the CSV filenames produced by Notebook 06 and must not
# change). DISPLAY_NAMES maps each key to the descriptive label used in
# every printed output, saved table, and figure in this notebook.
DISPLAY_NAMES = {
    "primary": "Market Only",
    "candidate": "Market + Google Trends",
    "ablation": "Market + Fear & Greed Index",
    "combined": "Market + Google Trends + Fear & Greed Index",
}

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15  # no fixed split was documented, going with this

RF_GRID = [
    {"n_estimators": 200, "max_depth": 3, "min_samples_split": 5, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 4, "min_samples_split": 5, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": 3, "min_samples_split": 10, "min_samples_leaf": 4, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": 2, "min_samples_split": 10, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"n_estimators": 800, "max_depth": 3, "min_samples_split": 15, "min_samples_leaf": 5, "max_features": "log2"},
    {"n_estimators": 300, "max_depth": 2, "min_samples_split": 20, "min_samples_leaf": 8, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 5, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": 0.5},
    {"n_estimators": 500, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
]

XGB_GRID = [
    {"n_estimators": 500, "max_depth": 2, "learning_rate": 0.01, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5, "reg_lambda": 1.0},
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5, "reg_lambda": 5.0},
    {"n_estimators": 500, "max_depth": 2, "learning_rate": 0.05, "subsample": 1.0, "colsample_bytree": 1.0, "min_child_weight": 10, "reg_lambda": 10.0},
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5, "reg_lambda": 5.0, "objective": "reg:pseudohubererror"},
    {"n_estimators": 500, "max_depth": 1, "learning_rate": 0.1, "subsample": 0.6, "colsample_bytree": 0.6, "min_child_weight": 10, "gamma": 0.5, "reg_lambda": 5.0},
    {"n_estimators": 500, "max_depth": 2, "learning_rate": 0.1, "subsample": 0.6, "colsample_bytree": 0.8, "min_child_weight": 10, "gamma": 1.0, "reg_alpha": 1.0, "reg_lambda": 5.0},
]

LSTM_GRID = [
    {"window": 5, "units": 16, "dropout": 0.2, "recurrent_dropout": 0.0, "lr": 0.001},
    {"window": 5, "units": 16, "dropout": 0.2, "recurrent_dropout": 0.2, "lr": 0.001},
    {"window": 5, "units": 16, "dropout": 0.3, "recurrent_dropout": 0.2, "lr": 0.001},
    {"window": 5, "units": 32, "dropout": 0.3, "recurrent_dropout": 0.2, "lr": 0.001},
    {"window": 5, "units": 32, "dropout": 0.3, "recurrent_dropout": 0.2, "lr": 0.0005},
    {"window": 7, "units": 16, "dropout": 0.2, "recurrent_dropout": 0.2, "lr": 0.001},
    {"window": 10, "units": 16, "dropout": 0.2, "recurrent_dropout": 0.2, "lr": 0.001},
    {"window": 14, "units": 16, "dropout": 0.2, "recurrent_dropout": 0.2, "lr": 0.001},
    {"window": 3, "units": 16, "dropout": 0.2, "recurrent_dropout": 0.1, "lr": 0.001},
]

LSTM_EPOCHS = 80
LSTM_BATCH = 32

CV_FOLDS = 3


## Feature Configuration Comparison

Each model in this notebook is trained and evaluated under four feature configurations, rather than once. Together these four configurations form the Feature Contribution Analysis for this dissertation:

| Configuration | Description |
|---|---|
| **Market Only** | Price and volume history alone -- the baseline configuration |
| **Market + Google Trends** | Market Only plus Google search interest |
| **Market + Fear & Greed Index** | Market Only plus the Crypto Fear & Greed Index |
| **Market + Google Trends + Fear & Greed Index** | Market Only plus both external signals combined |

Comparing these four configurations against one another isolates three questions directly: whether Google Trends contributes predictive value beyond market data alone, whether the Fear & Greed Index does the same, and whether combining both provides any additional value over either signal individually. This is the same underlying data produced in Notebook 06; every printed result, saved table, and figure in this notebook uses the descriptive names above.

## Loading the Feature Sets

Each of the four feature configurations is loaded and validated for both assets: target and price-reference columns present, chronological order preserved, and no duplicate dates. No features are recomputed at this stage.

In [10]:
def load_feature_set(asset, fset, price_col, horizon):
    fname = FEATURE_FILE_TEMPLATES[fset].format(a=asset.lower(), s=horizon_suffix_for(horizon))
    path = FEATURES_DIR / fname
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

    target_col = f"{price_col}_log_return_target_t_plus_{horizon}"
    level_col = f"{price_col}_level"
    assert target_col in df.columns
    assert level_col in df.columns
    assert df["Date"].is_monotonic_increasing
    assert not df["Date"].duplicated().any()

    print(f"h={horizon:2d}d {asset:9s} {DISPLAY_NAMES[fset]:38s} shape={df.shape} "
          f"{df['Date'].min().date()}->{df['Date'].max().date()} missing={df.isnull().sum().sum()}")
    return df


data = {}
for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        for fset in FEATURE_FILE_TEMPLATES:
            data[(horizon, asset, fset)] = load_feature_set(asset, fset, price_col, horizon)

print(f"\nloaded {len(data)} feature sets across {len(FORECAST_HORIZONS)} horizons")

h= 1d Bitcoin   Market Only                            shape=(1767, 17) 2021-07-02->2026-05-05 missing=0
h= 1d Bitcoin   Market + Google Trends                 shape=(1767, 21) 2021-07-02->2026-05-05 missing=0
h= 1d Bitcoin   Market + Fear & Greed Index            shape=(1767, 23) 2021-07-02->2026-05-05 missing=0
h= 1d Bitcoin   Market + Google Trends + Fear & Greed Index shape=(1767, 27) 2021-07-02->2026-05-05 missing=0
h= 1d Ethereum  Market Only                            shape=(1767, 17) 2021-07-02->2026-05-05 missing=0
h= 1d Ethereum  Market + Google Trends                 shape=(1767, 21) 2021-07-02->2026-05-05 missing=0
h= 1d Ethereum  Market + Fear & Greed Index            shape=(1767, 23) 2021-07-02->2026-05-05 missing=0
h= 1d Ethereum  Market + Google Trends + Fear & Greed Index shape=(1767, 27) 2021-07-02->2026-05-05 missing=0
h= 7d Bitcoin   Market Only                            shape=(1761, 18) 2021-07-02->2026-04-29 missing=0
h= 7d Bitcoin   Market + Google Trends       

In [11]:
# quick sanity check that the target really is the cumulative horizon-day
# return, not a single day's return that happens to fall that many days out
# (those are different things -- see Notebook 06's title cell for the full explanation)
# Run once, at horizon=1, as a representative case.
_check_horizon = 1
df = data[(_check_horizon, "Bitcoin", "primary")]
tgt = f"Bitcoin_Price_log_return_target_t_plus_{_check_horizon}"
level = "Bitcoin_Price_level"

# Recompute directly from this file's own price_level column: target at row t
# should equal log(price_level at row t+horizon) - log(price_level at row t)
recomputed = np.log(df[level].shift(-_check_horizon)) - np.log(df[level])
# Compare only where the recomputation has a value: the file's own price_level
# column is truncated for reasons unrelated to the 7-day target (confirmed
# separately -- the file's cutoff date is earlier than the point where the
# target would actually run out of future data), so shift(-FORECAST_HORIZON)
# on this already-truncated column alone correctly produces NaN for the last
# few rows even though the file's own target value there is valid and correct.
aligned = pd.concat([df[tgt], recomputed], axis=1, keys=["from_file", "recomputed"]).dropna()
ok = np.isclose(aligned["from_file"], aligned["recomputed"]).all()
print(f"target checks out ({len(aligned)} of {len(df)} rows compared, rest excluded due to the truncation note above): {ok}")
assert ok, f"target does not match the cumulative {_check_horizon}-day return - something's off"

target checks out (1766 of 1767 rows compared, rest excluded due to the truncation note above): True


## Chronological Train/Validation/Test Split

A single split function is applied uniformly across all asset/feature-configuration combinations, with assertions confirming no temporal overlap between partitions.

In [12]:
def split_data(df):
    n = len(df)
    train_end = int(n * TRAIN_FRAC)
    val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

    train = df.iloc[:train_end].reset_index(drop=True)
    val = df.iloc[train_end:val_end].reset_index(drop=True)
    test = df.iloc[val_end:].reset_index(drop=True)

    assert train["Date"].max() < val["Date"].min()
    assert val["Date"].max() < test["Date"].min()

    return train, val, test


splits = {}
for key, df in data.items():
    splits[key] = split_data(df)

for (horizon, asset, fset), (train, val, test) in splits.items():
    n = len(train) + len(val) + len(test)
    print(f"h={horizon:2d}d {asset:9s} {DISPLAY_NAMES[fset]:38s} train={len(train)} ({len(train)/n:.0%}) "
          f"val={len(val)} ({len(val)/n:.0%}) test={len(test)} ({len(test)/n:.0%})")

h= 1d Bitcoin   Market Only                            train=1236 (70%) val=265 (15%) test=266 (15%)
h= 1d Bitcoin   Market + Google Trends                 train=1236 (70%) val=265 (15%) test=266 (15%)
h= 1d Bitcoin   Market + Fear & Greed Index            train=1236 (70%) val=265 (15%) test=266 (15%)
h= 1d Bitcoin   Market + Google Trends + Fear & Greed Index train=1236 (70%) val=265 (15%) test=266 (15%)
h= 1d Ethereum  Market Only                            train=1236 (70%) val=265 (15%) test=266 (15%)
h= 1d Ethereum  Market + Google Trends                 train=1236 (70%) val=265 (15%) test=266 (15%)
h= 1d Ethereum  Market + Fear & Greed Index            train=1236 (70%) val=265 (15%) test=266 (15%)
h= 1d Ethereum  Market + Google Trends + Fear & Greed Index train=1236 (70%) val=265 (15%) test=266 (15%)
h= 7d Bitcoin   Market Only                            train=1232 (70%) val=264 (15%) test=265 (15%)
h= 7d Bitcoin   Market + Google Trends                 train=1232 (70%) val=264 (

## X/y Preparation

Naive uses only the lag-1 return. ARIMA is univariate on the return series itself, with no feature matrix at all. Random Forest, XGBoost, and LSTM use every remaining column except Date, the price-level column (kept only to reconstruct prices later, not used as a predictor), and the target.

In [13]:
# get_X_y() is imported above from src.models.data_prep -- see the imports cell.
print("get_X_y imported from src.models.data_prep")

get_X_y imported from src.models.data_prep


## Metrics

RMSE, MAE, and MAPE are calculated on reconstructed prices; directional accuracy is calculated on returns, since direction only has meaning there. MAPE is not calculated on returns, since log returns are frequently close to zero and percentage error becomes unstable as a result.

In [14]:
# rmse, mae, mape, dir_acc, r_squared, to_price, and evaluate() are imported above
# from src.evaluation.metrics -- see the imports cell and the markdown above for
# what each metric measures.
print("Evaluation metrics imported from src.evaluation.metrics")

Evaluation metrics imported from src.evaluation.metrics


## Naive Baseline

The Naive baseline predicts tomorrow's return as equal to today's -- no fitting is involved. Every other model in this notebook is measured against this simplest possible benchmark.

In [15]:
# run_naive() is imported above from src.models.baseline -- see the imports cell
# and the markdown above for the persistence rule itself.
print("run_naive imported from src.models.baseline")

run_naive imported from src.models.baseline


## ARIMA

The order is selected by AIC over a small grid (d=0, since the return series is already stationary). Forecasts are generated one day at a time, with the real observed value fed back in after each step rather than forecasting the whole test period blind -- this keeps ARIMA directly comparable to Random Forest and XGBoost, which naturally receive fresh lag information on every row.

In [16]:
# pick_arima_order(), arima_rolling(), and run_arima() are imported above from
# src.models.arima_model -- see the imports cell and the markdown above for the
# order-selection and rolling-forecast methodology.
print("ARIMA functions imported from src.models.arima_model")

ARIMA functions imported from src.models.arima_model


## Random Forest

Random Forest is tuned using TimeSeriesSplit on the training set only. The best configuration is then refit on the combined training and validation data and tested once. A separate, train-only fit is kept purely to report honest validation-set numbers, since by that point the "final" model has already seen the validation data itself.

In [17]:
# tune_rf() and run_random_forest() are imported above from src.models.tree_models
# -- see the imports cell. RF_GRID and CV_FOLDS (defined earlier in this notebook,
# in the configuration cell) remain the visible, authoritative hyperparameter
# search space -- passed explicitly into run_random_forest() below rather than
# relying on src's own internal default grid.
print("Random Forest functions imported from src.models.tree_models")

Random Forest functions imported from src.models.tree_models


## XGBoost

XGBoost uses early stopping against the validation set, never the test set. Once the best configuration and its stopped tree count are known, the model is refit on the combined training and validation data using that fixed tree count -- early stopping is not used again at this stage, since no held-out set remains once training and validation are combined.

In [18]:
# run_xgboost() is imported above from src.models.tree_models -- see the imports
# cell. XGB_GRID (defined earlier in this notebook) remains the visible,
# authoritative hyperparameter search space, passed explicitly below.
print("run_xgboost imported from src.models.tree_models")

run_xgboost imported from src.models.tree_models


## LSTM

The tabular features are reshaped into 5-day sequence windows. The scaler is fit on the training data only. Early stopping monitors validation loss directly, and the test set is never used in any way during training.

In [19]:
# make_sequences(), fit_lstm_once(), and run_lstm() are imported above from
# src.models.lstm_model -- see the imports cell and the markdown above for the
# windowing and architecture methodology. LSTM_GRID and LSTM_EPOCHS (defined
# earlier in this notebook) remain the visible, authoritative search space and
# training budget, passed explicitly into run_lstm() below.
print("LSTM functions imported from src.models.lstm_model")

LSTM functions imported from src.models.lstm_model


## Model Execution Across All Feature Configurations

All five models are executed for both assets across all four feature configurations (Market Only, Market + Google Trends, Market + Fear & Greed Index, and Market + Google Trends + Fear & Greed Index), completing the Feature Contribution Analysis introduced above. LSTM is the most computationally intensive model in this notebook due to per-configuration tuning.

In [20]:
# ARIMA and the persistence baseline are univariate and depend only on
# the asset's own price series, which is identical across all four
# feature sets (at a fixed horizon). Computing them once per asset per
# horizon, instead of once per feature set, removes four redundant,
# numerically identical experiments per asset/horizon without changing
# any result.

naive_by_asset = {}  # keyed by (horizon, asset)
arima_by_asset = {}  # keyed by (horizon, asset)

for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        train, val, test = splits[(horizon, asset, "primary")]
        naive_by_asset[(horizon, asset)] = run_naive(train, val, test, price_col, horizon=horizon)
        print(f"h={horizon:2d}d {asset}: fitting ARIMA once (shared across all feature sets)")
        arima_by_asset[(horizon, asset)] = run_arima(train, val, test, price_col, horizon=horizon)

h= 1d Bitcoin: fitting ARIMA once (shared across all feature sets)
  arima order: (1, 0, 0)
h= 1d Ethereum: fitting ARIMA once (shared across all feature sets)
  arima order: (1, 0, 2)
h= 7d Bitcoin: fitting ARIMA once (shared across all feature sets)
  arima order: (1, 0, 0)
h= 7d Ethereum: fitting ARIMA once (shared across all feature sets)
  arima order: (0, 0, 3)
h=14d Bitcoin: fitting ARIMA once (shared across all feature sets)
  arima order: (1, 0, 0)
h=14d Ethereum: fitting ARIMA once (shared across all feature sets)
  arima order: (3, 0, 0)


### Standardised Prediction Output (Return-Space and Price-Space)

One function builds every model's saved prediction rows, in both return-space and reconstructed price-space, so no model gets a bespoke saving path. `price_level` is looked up by `event_date` rather than by row position -- this is what makes the alignment correct regardless of a model's exact row count for a given split, rather than assuming every model produces the same number of rows.

In [21]:
def build_prediction_rows(result: dict, asset: str, model_name: str, feature_set_key: str,
                           split_data_map: dict, price_col: str, horizon: int,
                           representation: str = "Original") -> pd.DataFrame:
    """Standardise one model's predictions into the full return-space + price-space schema.

    price_level is looked up by event_date against the split's own dataframe,
    not by row position -- this stays correct even if a model's prediction
    count differs from the raw split length for any reason. `horizon` is
    included as its own column so predictions across all three forecast
    horizons can be filtered and compared directly from one combined file.
    `representation` distinguishes the original engineered features from the
    autoencoder-encoded branch -- "Original" for every result already in
    this notebook before the autoencoder was added, "Autoencoder-Encoded"
    for the new branch, so both can be filtered and compared directly from
    the same combined predictions file.
    """
    level_col = f"{price_col}_level"
    rows = []

    for split_name, pred_df in result["predictions"].items():
        source_df = split_data_map[split_name]
        price_lookup = source_df.set_index("Date")[level_col]

        pdf = pred_df.copy()
        pdf["price_level"] = pdf["event_date"].map(price_lookup).values
        pdf["actual_price"] = pdf["price_level"] * np.exp(pdf["actual"])
        pdf["predicted_price"] = pdf["price_level"] * np.exp(pdf["predicted"])
        pdf["price_residual"] = pdf["actual_price"] - pdf["predicted_price"]
        pdf = pdf.rename(columns={"actual": "actual_return", "predicted": "predicted_return"})
        pdf["return_residual"] = pdf["actual_return"] - pdf["predicted_return"]
        pdf["horizon"], pdf["asset"], pdf["model"], pdf["split"], pdf["feature_set"], pdf["representation"] = horizon, asset, model_name, split_name, feature_set_key, representation

        rows.append(pdf[[
            "horizon", "event_date", "asset", "model", "split", "feature_set", "representation", "price_level",
            "actual_return", "predicted_return", "return_residual",
            "actual_price", "predicted_price", "price_residual",
        ]])

    return pd.concat(rows, ignore_index=True)

In [22]:
pred_rows, metric_rows, importance_rows, log_rows = [], [], [], []
xgboost_results = {}  # retained for the Phase 4 XGBoost diagnostic below -- not used for retraining
t0 = time.time()

for (horizon, asset, fset), df in data.items():
    price_col = ASSETS[asset]
    train, val, test = splits[(horizon, asset, fset)]
    print(f"\n=== h={horizon}d / {asset} / {DISPLAY_NAMES[fset]} ===")

    runners = {
        "Naive": lambda: naive_by_asset[(horizon, asset)],
        "ARIMA": lambda: arima_by_asset[(horizon, asset)],
        "Random Forest": lambda: run_random_forest(train, val, test, price_col, horizon=horizon, grid=RF_GRID, cv_folds=CV_FOLDS),
        "XGBoost": lambda: run_xgboost(train, val, test, price_col, horizon=horizon, grid=XGB_GRID),
        "LSTM": lambda: run_lstm(train, val, test, price_col, horizon=horizon, grid=LSTM_GRID, epochs=LSTM_EPOCHS),
    }

    for model_name, fn in runners.items():
        t1 = time.time()
        status = "success"
        try:
            result = fn()
        except Exception as e:
            print(f"  {model_name} failed: {e}")
            status = f"failed: {e}"
            result = None
        elapsed = time.time() - t1

        # Naive and ARIMA are precomputed once per asset/horizon above; report
        # zero incremental cost here rather than a misleading duration.
        if model_name in ("Naive", "ARIMA"):
            elapsed = 0.0

        log_rows.append({"horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset], "model": model_name,
                          "representation": "Original", "status": status, "seconds": round(elapsed, 1)})

        if result is None:
            continue

        split_data_map = {"validation": val, "test": test}
        pred_rows.append(build_prediction_rows(result, asset, model_name, DISPLAY_NAMES[fset], split_data_map, price_col, horizon=horizon, representation="Original"))

        if model_name == "XGBoost":
            xgboost_results[(horizon, asset, fset)] = result

        for split_name, m in result["metrics"].items():
            metric_rows.append({"horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset], "model": model_name,
                                 "representation": "Original", "split": split_name, "seconds": round(elapsed, 1), "status": status, **m})

        # File-naming prefix intentionally keeps the internal key (fset),
        # not the descriptive DISPLAY_NAMES label -- the display names
        # contain spaces and a "+" character that are unsuitable for
        # filesystem-safe filenames. This affects saved artefact
        # filenames only, never a printed output, table, or figure.
        # horizon_suffix_for(horizon) included so each horizon's artefacts
        # never overwrite another horizon's.
        prefix = f"{asset.lower()}_{fset}{horizon_suffix_for(horizon)}_{model_name.lower().replace(' ', '_')}"

        if model_name in ("Random Forest", "XGBoost"):
            joblib.dump(result["model"], MODELS_DIR / f"{prefix}.joblib")
            imp = result["importance"].copy()
            imp["horizon"], imp["asset"], imp["feature_set"], imp["model"], imp["representation"] = horizon, asset, DISPLAY_NAMES[fset], model_name, "Original"
            importance_rows.append(imp)

        if model_name == "LSTM":
            result["model"].save(MODELS_DIR / f"{prefix}.keras")
            joblib.dump(result["scaler"], MODELS_DIR / f"{prefix}_scaler.joblib")
            json.dump(result["config"], open(MODELS_DIR / f"{prefix}_config.json", "w"))

        print(f"  {model_name}: {elapsed:.1f}s")

print(f"\ndone in {(time.time()-t0)/60:.1f} min")


=== h=1d / Bitcoin / Market Only ===
  Naive: 0.0s
  ARIMA: 0.0s
  rf best params: {'n_estimators': 300, 'max_depth': 2, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': 'sqrt'} (cv rmse=0.02762)
  Random Forest: 10.0s
  xgb best params: {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 5.0, 'objective': 'reg:pseudohubererror'}, stopped at 25 trees (val rmse=0.02284)
  XGBoost: 0.2s
  lstm configuration 1/9: training...
  lstm configuration 1/9: {'window': 5, 'units': 16, 'dropout': 0.2, 'recurrent_dropout': 0.0, 'lr': 0.001} -> val_rmse=0.02819 (4.7s) [completed]
  lstm configuration 2/9: training...
  lstm configuration 2/9: {'window': 5, 'units': 16, 'dropout': 0.2, 'recurrent_dropout': 0.2, 'lr': 0.001} -> val_rmse=0.02752 (5.6s) [completed]
  lstm configuration 3/9: training...
  lstm configuration 3/9: {'window': 5, 'units': 16, 'dropout': 0.3, 'recurrent_dropout': 0.2, 'lr'

## Feature Extraction (Autoencoder) -- An Additional Experimental Branch

Per the supervisor's request, this section adds Autoencoder-based feature extraction as a **second, parallel experimental branch** -- it does not replace the engineered features used everywhere above. Every result already produced in this notebook (tagged `representation = "Original"`) stays exactly as it was; this section adds a second set of results tagged `representation = "Autoencoder-Encoded"`, so both can be compared directly, model for model, horizon for horizon, feature configuration for feature configuration.

**The autoencoder itself is trained in a separate notebook: `06b_autoencoder_feature_extraction.ipynb`.** This section was originally built as inline cells here, but was split out after those cells were found to have been silently skipped during a full run of this notebook -- everything downstream still executed successfully using only the original features, with no error to signal that the encoded branch had never actually run. Running the autoencoder as its own notebook, with its own clear pass/fail verification, means that risk no longer exists: this section below only *loads* the already-encoded, already-verified CSV files that notebook produces.

```
Feature Engineering (06)
        |
        v
Feature Extraction / Autoencoder (06b, separate notebook, run first)
        |
        v
   [this section loads the saved encoded CSVs]
        |
        v
Random Forest / XGBoost / LSTM (both branches, below)
```

**Prerequisite: run `06b_autoencoder_feature_extraction.ipynb` before this section.** The cell below will raise a clear `FileNotFoundError` naming the missing file if it has not been run.

**Scope.** The autoencoder is trained separately for every (horizon, asset, feature configuration) combination in `06b`. **ARIMA and Naive are excluded from the encoded branch**: ARIMA is univariate on the price series alone and never receives the feature matrix in the first place, and Naive has no features or parameters at all -- "encoded features" does not apply to either, structurally, not as a judgement call.

**Leakage discipline** is enforced in `06b`, not repeated here: the autoencoder's weights are fit only on training-split data; the validation split is used only for `EarlyStopping`'s monitored loss; the test split is only ever encoded through the already-fitted encoder. See that notebook for the full explanation and its own verification cell.

In [23]:
# Loads the encoded CSVs produced by 06b_autoencoder_feature_extraction.ipynb --
# does not train anything here. If any expected file is missing, this raises
# immediately with the exact path, rather than silently producing an empty
# or partial encoded_data dict that later cells would fail on less clearly.

AE_DATA_DIR = PROJECT_ROOT / "results" / "autoencoder" / "encoded_features"
ENCODING_DIM = 8  # must match 06b's ENCODING_DIM -- checked below, not just assumed

encoded_data = {}  # keyed by (horizon, asset, fset) -> {"train": df, "val": df, "test": df}

for (horizon, asset, fset) in data:
    price_col = ASSETS[asset]
    prefix = f"{asset.lower()}_{fset}{horizon_suffix_for(horizon)}"

    split_frames = {}
    for split_key, split_filename in [("train", "train"), ("val", "validation"), ("test", "test")]:
        path = AE_DATA_DIR / f"{prefix}_encoded_{split_filename}.csv"
        if not path.exists():
            raise FileNotFoundError(
                f"{path} not found. Run 06b_autoencoder_feature_extraction.ipynb "
                f"first, for horizon={horizon}, asset={asset}, feature_set={fset}."
            )
        edf = pd.read_csv(path, parse_dates=["Date"])
        encoded_cols = [c for c in edf.columns if c.startswith("encoded_")]
        assert len(encoded_cols) == ENCODING_DIM, (
            f"{path} has {len(encoded_cols)} encoded columns, expected {ENCODING_DIM} -- "
            "06b's ENCODING_DIM may have changed since this file was produced."
        )
        split_frames[split_key] = edf

    encoded_data[(horizon, asset, fset)] = split_frames

print(f"Loaded encoded features for {len(encoded_data)} (horizon, asset, feature_set) combinations "
      f"from {AE_DATA_DIR}")

Loaded encoded features for 24 (horizon, asset, feature_set) combinations from /Users/mas364/Desktop/AI_Crypto_Forecasting/results/autoencoder/encoded_features


### Training Random Forest, XGBoost, and LSTM on the Encoded Features

Reuses `run_random_forest`, `run_xgboost`, `run_lstm`, and `build_prediction_rows` completely unchanged -- the encoded dataframes have the exact same structure as the original feature sets (`Date`, feature columns, price-level reference, target), so every function that worked on the original 14-27 engineered features works identically on the 8 encoded ones, with no special-casing required. Results are appended to the same `pred_rows`, `metric_rows`, `log_rows`, and `importance_rows` lists used above, tagged `representation = "Autoencoder-Encoded"`, so the combined predictions and metrics files contain both branches directly comparable, without a separate parallel set of output files to reconcile.

In [24]:
t0_ae = time.time()

for (horizon, asset, fset), enc in encoded_data.items():
    price_col = ASSETS[asset]
    train_enc, val_enc, test_enc = enc["train"], enc["val"], enc["test"]
    print(f"\n=== [Encoded] h={horizon}d / {asset} / {DISPLAY_NAMES[fset]} ===")

    runners = {
        "Random Forest": lambda: run_random_forest(train_enc, val_enc, test_enc, price_col, horizon=horizon, grid=RF_GRID, cv_folds=CV_FOLDS),
        "XGBoost": lambda: run_xgboost(train_enc, val_enc, test_enc, price_col, horizon=horizon, grid=XGB_GRID),
        "LSTM": lambda: run_lstm(train_enc, val_enc, test_enc, price_col, horizon=horizon, grid=LSTM_GRID, epochs=LSTM_EPOCHS),
    }

    for model_name, fn in runners.items():
        t1 = time.time()
        status = "success"
        try:
            result = fn()
        except Exception as e:
            print(f"  {model_name} failed: {e}")
            status = f"failed: {e}"
            result = None
        elapsed = time.time() - t1

        log_rows.append({"horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset], "model": model_name,
                          "representation": "Autoencoder-Encoded", "status": status, "seconds": round(elapsed, 1)})

        if result is None:
            continue

        split_data_map = {"validation": val_enc, "test": test_enc}
        pred_rows.append(build_prediction_rows(result, asset, model_name, DISPLAY_NAMES[fset], split_data_map,
                                                 price_col, horizon=horizon, representation="Autoencoder-Encoded"))

        for split_name, m in result["metrics"].items():
            metric_rows.append({"horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset], "model": model_name,
                                 "representation": "Autoencoder-Encoded", "split": split_name,
                                 "seconds": round(elapsed, 1), "status": status, **m})

        prefix = f"{asset.lower()}_{fset}{horizon_suffix_for(horizon)}_{model_name.lower().replace(' ', '_')}_encoded"

        if model_name in ("Random Forest", "XGBoost"):
            joblib.dump(result["model"], MODELS_DIR / f"{prefix}.joblib")
            imp = result["importance"].copy()
            imp["horizon"], imp["asset"], imp["feature_set"], imp["model"], imp["representation"] = (
                horizon, asset, DISPLAY_NAMES[fset], model_name, "Autoencoder-Encoded"
            )
            importance_rows.append(imp)

        if model_name == "LSTM":
            result["model"].save(MODELS_DIR / f"{prefix}.keras")
            joblib.dump(result["scaler"], MODELS_DIR / f"{prefix}_scaler.joblib")
            json.dump(result["config"], open(MODELS_DIR / f"{prefix}_config.json", "w"))

        print(f"  {model_name}: {elapsed:.1f}s")

print(f"\nEncoded-feature training done in {(time.time()-t0_ae)/60:.1f} min")


=== [Encoded] h=1d / Bitcoin / Market Only ===
  rf best params: {'n_estimators': 300, 'max_depth': 2, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': 'sqrt'} (cv rmse=0.02766)
  Random Forest: 8.6s
  xgb best params: {'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.05, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 10, 'reg_lambda': 10.0}, stopped at 36 trees (val rmse=0.02282)
  XGBoost: 0.1s
  lstm configuration 1/9: training...
  lstm configuration 1/9: {'window': 5, 'units': 16, 'dropout': 0.2, 'recurrent_dropout': 0.0, 'lr': 0.001} -> val_rmse=0.02476 (4.8s) [completed]
  lstm configuration 2/9: training...
  lstm configuration 2/9: {'window': 5, 'units': 16, 'dropout': 0.2, 'recurrent_dropout': 0.2, 'lr': 0.001} -> val_rmse=0.02469 (8.0s) [completed]
  lstm configuration 3/9: training...
  lstm configuration 3/9: {'window': 5, 'units': 16, 'dropout': 0.3, 'recurrent_dropout': 0.2, 'lr': 0.001} -> val_rmse=0.02524 (5.9s) [completed]
  lstm

In [25]:
all_preds = pd.concat(pred_rows, ignore_index=True)
all_preds.to_csv(PRED_DIR / "all_model_predictions.csv", index=False)
print(f"saved predictions: {len(all_preds)} rows")

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(METRICS_DIR / "model_development_metrics.csv", index=False)

if importance_rows:
    pd.concat(importance_rows, ignore_index=True).to_csv(METRICS_DIR / "feature_importance.csv", index=False)

log_df = pd.DataFrame(log_rows)
log_df.to_csv(METRICS_DIR / "run_log.csv", index=False)

print("all saved")
display(all_preds.head())

saved predictions: 101568 rows
all saved


,horizon,event_date,asset,model,split,feature_set,representation,price_level,actual_return,predicted_return,return_residual,actual_price,predicted_price,price_residual
0,1,2024-11-21,Bitcoin,Naive,validation,Market Only,Original,98509.118591,0.004238,0.044548,-0.040310,98927.494946,102996.743148,-4069.248202
1,1,2024-11-22,Bitcoin,Naive,validation,Market Only,Original,98927.494946,-0.012696,0.004238,-0.016934,97679.463816,99347.648179,-1668.184363
2,1,2024-11-23,Bitcoin,Naive,validation,Market Only,Original,97679.463816,0.003439,-0.012696,0.016135,98015.935529,96447.177367,1568.758162
3,1,2024-11-24,Bitcoin,Naive,validation,Market Only,Original,98015.935529,-0.052480,0.003439,-0.055919,93004.700931,98353.566269,-5348.865338
4,1,2024-11-25,Bitcoin,Naive,validation,Market Only,Original,93004.700931,-0.011603,-0.052480,0.040877,91931.830773,88249.674388,3682.156385


# Saving Trained Model Artefacts

This notebook acts as the experiment orchestration layer for model development: it loads data, runs training and tuning, and records predictions, metrics, and diagnostics. The reusable training and evaluation logic itself lives in `src/models/`, which this notebook calls but does not redefine.

This section persists the trained model artefacts produced above to a structured, project-level location (`models/trained/`), separate from the experimental results already saved to `results/modeling/`. Persisting these artefacts supports reproducibility -- a specific trained model can be reloaded exactly as it was, without repeating tuning or training -- and provides the artefact information a later inference pipeline would need.

Persistence here does not change any experimental result. No model is retrained, no prediction is recomputed, and no metric changes as a result of this section; it only reorganises and documents already-trained objects, which the training loop above already saved once to `results/modeling/fitted_models/`. Saving a model artefact also does not, by itself, mean the project is ready for production deployment -- a working inference pipeline and a deployed application are each separate pieces of work, addressed elsewhere.

In [26]:
import datetime
import sklearn
import xgboost as xgb_pkg
import tensorflow as tf_pkg
import statsmodels

# Human-readable, filesystem-safe directory names for each feature configuration.
# Kept separate from DISPLAY_NAMES above (which is for printed/tabular output,
# and contains spaces and "+" characters unsuitable for directory names).
DIR_NAMES = {
    "primary": "market_only",
    "candidate": "market_google_trends",
    "ablation": "market_fear_greed",
    "combined": "market_all_external",
}

# Base path only -- each horizon gets its own subfolder further down
# (MODELS_ROOT / f"horizon_{horizon}day" / ...), so this root itself
# does not need a horizon baked into it.
MODELS_ROOT = PROJECT_ROOT / "models" / "trained"

LIBRARY_VERSIONS = {
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb_pkg.__version__,
    "tensorflow": tf_pkg.__version__,
    "statsmodels": statsmodels.__version__,
    "joblib": joblib.__version__,
}


def json_safe(obj):
    """Recursively convert numpy/pandas/Path/datetime values into plain JSON-serialisable types."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp, datetime.date, datetime.datetime)):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    return obj


print(f"Artefacts will be saved under: {MODELS_ROOT}")
print(f"Library versions recorded: {LIBRARY_VERSIONS}")

Artefacts will be saved under: /Users/mas364/Desktop/AI_Crypto_Forecasting/models/trained
Library versions recorded: {'scikit_learn': '1.9.0', 'xgboost': '3.4.0', 'tensorflow': '2.21.0', 'statsmodels': '0.14.6', 'joblib': '1.5.3'}


### Persisting Random Forest, XGBoost, and LSTM Artefacts

Random Forest, XGBoost, and LSTM already have their fitted objects saved once by the training loop above, to `results/modeling/fitted_models/`. Those files are loaded back here (not retrained) and re-saved into the structured `models/trained/{cryptocurrency}/{feature_configuration}/` layout, alongside a metadata JSON recording the exact feature order, target, date ranges, random seed, and library versions used.

Naive requires no persisted artefact, since it has no trainable parameters. ARIMA is handled separately below, since the fitted statsmodels object is not currently returned by `run_arima()`.

In [27]:
manifest_entries = []
skipped_artefacts = []

for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        for fset in FEATURE_FILE_TEMPLATES:
            train, val, test = splits[(horizon, asset, fset)]
            X_train, y_train = get_X_y(train, price_col, horizon=horizon)

            # horizon folded into the path so each horizon's persisted
            # artefacts live separately, never overwriting another horizon's
            target_dir = MODELS_ROOT / f"horizon_{horizon}day" / asset.lower() / DIR_NAMES[fset]
            target_dir.mkdir(parents=True, exist_ok=True)

            base_meta = {
                "horizon": horizon,
                "cryptocurrency": asset.lower(),
                "feature_set_key": fset,
                "feature_set_name": DIR_NAMES[fset],
                "target": f"{price_col}_log_return_target_t_plus_{horizon}",
                "feature_names": X_train.columns.tolist(),
                "random_seed": SEED,
                "library_versions": LIBRARY_VERSIONS,
                "training_period": [str(train["Date"].min().date()), str(train["Date"].max().date())],
                "validation_period": [str(val["Date"].min().date()), str(val["Date"].max().date())],
                "test_period": [str(test["Date"].min().date()), str(test["Date"].max().date())],
            }

            old_prefix = f"{asset.lower()}_{fset}{horizon_suffix_for(horizon)}"

            # --- Random Forest ---
            rf_old = MODELS_DIR / f"{old_prefix}_random_forest.joblib"
            if rf_old.exists():
                model = joblib.load(rf_old)
                model_path = target_dir / "random_forest.joblib"
                joblib.dump(model, model_path)
                meta = {**base_meta, "model_family": "random_forest",
                         "model_path": str(model_path.relative_to(PROJECT_ROOT)), "scaler_path": None}
                meta_path = target_dir / "random_forest_metadata.json"
                json.dump(json_safe(meta), open(meta_path, "w"), indent=2)
                manifest_entries.append({
                    "horizon": horizon, "cryptocurrency": asset.lower(), "model_family": "random_forest",
                    "feature_set_key": fset, "feature_set_name": DIR_NAMES[fset],
                    "model_path": str(model_path.relative_to(PROJECT_ROOT)), "scaler_path": None,
                    "metadata_path": str(meta_path.relative_to(PROJECT_ROOT)),
                    "model_format": "joblib", "feature_count": len(X_train.columns),
                    "target": base_meta["target"],
                })
                print(f"saved: {asset}/{DIR_NAMES[fset]}/random_forest.joblib")
            else:
                skipped_artefacts.append((asset, fset, "random_forest", "source artefact not found in results/modeling/fitted_models/"))

            # --- XGBoost ---
            xgb_old = MODELS_DIR / f"{old_prefix}_xgboost.joblib"
            if xgb_old.exists():
                model = joblib.load(xgb_old)
                model_path = target_dir / "xgboost.joblib"
                joblib.dump(model, model_path)
                meta = {**base_meta, "model_family": "xgboost",
                         "model_path": str(model_path.relative_to(PROJECT_ROOT)), "scaler_path": None}
                meta_path = target_dir / "xgboost_metadata.json"
                json.dump(json_safe(meta), open(meta_path, "w"), indent=2)
                manifest_entries.append({
                    "horizon": horizon, "cryptocurrency": asset.lower(), "model_family": "xgboost",
                    "feature_set_key": fset, "feature_set_name": DIR_NAMES[fset],
                    "model_path": str(model_path.relative_to(PROJECT_ROOT)), "scaler_path": None,
                    "metadata_path": str(meta_path.relative_to(PROJECT_ROOT)),
                    "model_format": "joblib", "feature_count": len(X_train.columns),
                    "target": base_meta["target"],
                })
                print(f"saved: {asset}/{DIR_NAMES[fset]}/xgboost.joblib")
            else:
                skipped_artefacts.append((asset, fset, "xgboost", "source artefact not found in results/modeling/fitted_models/"))

            # --- LSTM (model + scaler + sequence config) ---
            lstm_old = MODELS_DIR / f"{old_prefix}_lstm.keras"
            scaler_old = MODELS_DIR / f"{old_prefix}_lstm_scaler.joblib"
            config_old = MODELS_DIR / f"{old_prefix}_lstm_config.json"
            if lstm_old.exists() and scaler_old.exists():
                lstm_model = keras.models.load_model(lstm_old)
                model_path = target_dir / "lstm.keras"
                lstm_model.save(model_path)

                scaler = joblib.load(scaler_old)
                scaler_path = target_dir / "lstm_scaler.joblib"
                joblib.dump(scaler, scaler_path)

                lstm_config = json.load(open(config_old)) if config_old.exists() else None

                meta = {**base_meta, "model_family": "lstm",
                         "model_path": str(model_path.relative_to(PROJECT_ROOT)),
                         "scaler_path": str(scaler_path.relative_to(PROJECT_ROOT)),
                         "sequence_config": lstm_config}
                meta_path = target_dir / "lstm_metadata.json"
                json.dump(json_safe(meta), open(meta_path, "w"), indent=2)
                manifest_entries.append({
                    "horizon": horizon, "cryptocurrency": asset.lower(), "model_family": "lstm",
                    "feature_set_key": fset, "feature_set_name": DIR_NAMES[fset],
                    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
                    "scaler_path": str(scaler_path.relative_to(PROJECT_ROOT)),
                    "metadata_path": str(meta_path.relative_to(PROJECT_ROOT)),
                    "model_format": "keras", "feature_count": len(X_train.columns),
                    "target": base_meta["target"],
                })
                print(f"saved: {asset}/{DIR_NAMES[fset]}/lstm.keras (+ scaler)")
            else:
                skipped_artefacts.append((asset, fset, "lstm", "source artefact not found in results/modeling/fitted_models/"))

    print(f"\n{len(manifest_entries)} artefacts persisted so far.")
    if skipped_artefacts:
        print(f"{len(skipped_artefacts)} skipped (source not found):")
        for s in skipped_artefacts:
            print(" ", s)

saved: Bitcoin/market_only/random_forest.joblib
saved: Bitcoin/market_only/xgboost.joblib
saved: Bitcoin/market_only/lstm.keras (+ scaler)
saved: Bitcoin/market_google_trends/random_forest.joblib
saved: Bitcoin/market_google_trends/xgboost.joblib
saved: Bitcoin/market_google_trends/lstm.keras (+ scaler)
saved: Bitcoin/market_fear_greed/random_forest.joblib
saved: Bitcoin/market_fear_greed/xgboost.joblib
saved: Bitcoin/market_fear_greed/lstm.keras (+ scaler)
saved: Bitcoin/market_all_external/random_forest.joblib
saved: Bitcoin/market_all_external/xgboost.joblib
saved: Bitcoin/market_all_external/lstm.keras (+ scaler)
saved: Ethereum/market_only/random_forest.joblib
saved: Ethereum/market_only/xgboost.joblib
saved: Ethereum/market_only/lstm.keras (+ scaler)
saved: Ethereum/market_google_trends/random_forest.joblib
saved: Ethereum/market_google_trends/xgboost.joblib
saved: Ethereum/market_google_trends/lstm.keras (+ scaler)
saved: Ethereum/market_fear_greed/random_forest.joblib
saved: Et

### ARIMA: Configuration Metadata Only

`run_arima()` in `src/models/arima_model.py` selects an order and produces rolling forecasts, but does not return the fitted `statsmodels` results object -- only the selected `(p, 0, q)` order is available afterward. No fitted ARIMA model can therefore be persisted here; retraining ARIMA solely to obtain a savable object would violate the requirement not to retrain or redesign models for the purpose of persistence.

What genuinely is available -- the selected order, target definition, and the date ranges used -- is saved as configuration metadata instead, with the limitation stated explicitly rather than implied.

In [28]:
for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        target_dir = MODELS_ROOT / f"horizon_{horizon}day" / asset.lower() / DIR_NAMES["primary"]
        target_dir.mkdir(parents=True, exist_ok=True)
        train, val, test = splits[(horizon, asset, "primary")]

        arima_order = arima_by_asset[(horizon, asset)].get("order") if (horizon, asset) in arima_by_asset else None

        meta = {
            "horizon": horizon,
            "cryptocurrency": asset.lower(),
            "model_family": "arima",
            "feature_set_key": "primary",
            "feature_set_name": DIR_NAMES["primary"],
            "target": f"{price_col}_log_return",
            "feature_names": None,
            "selected_order": arima_order,
            "model_path": None,
            "scaler_path": None,
            "training_period": [str(train["Date"].min().date()), str(train["Date"].max().date())],
            "validation_period": [str(val["Date"].min().date()), str(val["Date"].max().date())],
            "test_period": [str(test["Date"].min().date()), str(test["Date"].max().date())],
            "random_seed": SEED,
            "library_versions": LIBRARY_VERSIONS,
            "limitation": (
                "The fitted statsmodels ARIMA object is not returned by run_arima() / "
                "arima_rolling() -- only the selected (p, d, q) order is available. "
                "No fitted model file is saved for ARIMA; the order is preserved here "
                "as configuration metadata only."
            ),
        }
        meta_path = target_dir / "arima_metadata.json"
        json.dump(json_safe(meta), open(meta_path, "w"), indent=2)

        manifest_entries.append({
            "horizon": horizon, "cryptocurrency": asset.lower(), "model_family": "arima",
            "feature_set_key": "primary", "feature_set_name": DIR_NAMES["primary"],
            "model_path": None, "scaler_path": None,
            "metadata_path": str(meta_path.relative_to(PROJECT_ROOT)),
            "model_format": None, "feature_count": None, "target": meta["target"],
        })
        print(f"h={horizon}d saved metadata only: {asset}/{DIR_NAMES['primary']}/arima_metadata.json (order={arima_order})")

print(f"\nTotal manifest entries: {len(manifest_entries)}")

h=1d saved metadata only: Bitcoin/market_only/arima_metadata.json (order=(1, 0, 0))
h=1d saved metadata only: Ethereum/market_only/arima_metadata.json (order=(1, 0, 2))
h=7d saved metadata only: Bitcoin/market_only/arima_metadata.json (order=(1, 0, 0))
h=7d saved metadata only: Ethereum/market_only/arima_metadata.json (order=(0, 0, 3))
h=14d saved metadata only: Bitcoin/market_only/arima_metadata.json (order=(1, 0, 0))
h=14d saved metadata only: Ethereum/market_only/arima_metadata.json (order=(3, 0, 0))

Total manifest entries: 78


### Naive (Persistence) Baseline: Metadata Only

`run_naive()` in `src/models/baseline.py` implements a rule -- predict tomorrow's return as equal to today's observed return -- with no fitted parameters and no model object of any kind. Its role in the model comparison, its predictions, and its evaluation are unchanged by this section.

No model file is created for the Naive baseline, since there is no fitted object to save; creating one would misrepresent a parameterless rule as a trained model. What is saved instead is lightweight metadata describing the rule itself and the data periods it was evaluated on, for the same reproducibility reasons the other models' metadata is saved.

In [29]:
for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        target_dir = MODELS_ROOT / f"horizon_{horizon}day" / asset.lower() / DIR_NAMES["primary"]
        target_dir.mkdir(parents=True, exist_ok=True)
        train, val, test = splits[(horizon, asset, "primary")]

        reference_col = f"{price_col}_log_return" if horizon == 1 else f"{price_col}_trailing_{horizon}day_return"

        meta = {
            "horizon": horizon,
            "cryptocurrency": asset.lower(),
            "model_family": "naive_persistence",
            "feature_set_key": "primary",
            "feature_set_name": DIR_NAMES["primary"],
            "target": f"{price_col}_log_return_target_t_plus_{horizon}",
            "rule": f"prediction = observed {reference_col} -- no fitted parameters",
            "feature_names": None,
            "model_path": None,
            "scaler_path": None,
            "training_period": [str(train["Date"].min().date()), str(train["Date"].max().date())],
            "validation_period": [str(val["Date"].min().date()), str(val["Date"].max().date())],
            "test_period": [str(test["Date"].min().date()), str(test["Date"].max().date())],
            "random_seed": SEED,
            "limitation": (
                "Naive is a parameterless persistence rule, not a trained model -- "
                "there is no fitted object to serialise. This metadata records the "
                "rule and evaluation periods only."
            ),
        }
        meta_path = target_dir / "naive_persistence_metadata.json"
        json.dump(json_safe(meta), open(meta_path, "w"), indent=2)

        manifest_entries.append({
            "horizon": horizon, "cryptocurrency": asset.lower(), "model_family": "naive_persistence",
            "feature_set_key": "primary", "feature_set_name": DIR_NAMES["primary"],
            "model_path": None, "scaler_path": None,
            "metadata_path": str(meta_path.relative_to(PROJECT_ROOT)),
            "model_format": None, "feature_count": None, "target": meta["target"],
        })
        print(f"h={horizon}d saved metadata only: {asset}/{DIR_NAMES['primary']}/naive_persistence_metadata.json")

print(f"\nTotal manifest entries: {len(manifest_entries)}")

h=1d saved metadata only: Bitcoin/market_only/naive_persistence_metadata.json
h=1d saved metadata only: Ethereum/market_only/naive_persistence_metadata.json
h=7d saved metadata only: Bitcoin/market_only/naive_persistence_metadata.json
h=7d saved metadata only: Ethereum/market_only/naive_persistence_metadata.json
h=14d saved metadata only: Bitcoin/market_only/naive_persistence_metadata.json
h=14d saved metadata only: Ethereum/market_only/naive_persistence_metadata.json

Total manifest entries: 84


### Artefact Manifest

One machine-readable manifest listing every saved artefact, with paths relative to the project root so a later pipeline or deployment application can locate them regardless of machine.

In [30]:
manifest_path = MODELS_ROOT / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(json_safe(manifest_entries), f, indent=2)

print(f"Manifest saved: {manifest_path}")
print(f"Entries: {len(manifest_entries)}")

manifest_df = pd.DataFrame(manifest_entries)
display(manifest_df[["cryptocurrency", "model_family", "feature_set_name", "model_format", "feature_count"]])

Manifest saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/models/trained/manifest.json
Entries: 84


,cryptocurrency,model_family,feature_set_name,model_format,feature_count
0,bitcoin,random_forest,market_only,joblib,14.0
1,bitcoin,xgboost,market_only,joblib,14.0
2,bitcoin,lstm,market_only,keras,14.0
3,bitcoin,random_forest,market_google_trends,joblib,18.0
4,bitcoin,xgboost,market_google_trends,joblib,18.0
...,...,...,...,...,...
79,ethereum,naive_persistence,market_only,NaN,NaN
80,bitcoin,naive_persistence,market_only,NaN,NaN
81,ethereum,naive_persistence,market_only,NaN,NaN
82,bitcoin,naive_persistence,market_only,NaN,NaN


# Artefact Reload Verification

Reloads one saved artefact from each model family actually present, directly from the manifest -- not from variables already held in notebook memory. This confirms the saved artefacts are independently usable without rerunning any training above; it does not recompute any forecast or metric.

In [31]:
manifest = json.load(open(MODELS_ROOT / "manifest.json"))

verified_families = set()

for entry in manifest:
    family = entry["model_family"]
    if family in verified_families:
        continue

    if entry["model_path"] is None:
        print(f"{family}: metadata-only artefact (no fitted model to reload) -- expected, see {family}_metadata.json")
        verified_families.add(family)
        continue

    model_path = PROJECT_ROOT / entry["model_path"]

    if entry["model_format"] == "joblib":
        reloaded = joblib.load(model_path)
        print(f"{family}: reloaded {type(reloaded).__name__} from {model_path.name} -- OK")

    elif entry["model_format"] == "keras":
        reloaded = keras.models.load_model(model_path)
        print(f"{family}: reloaded Keras model ({reloaded.count_params()} params) from {model_path.name} -- OK")
        if entry["scaler_path"]:
            reloaded_scaler = joblib.load(PROJECT_ROOT / entry["scaler_path"])
            print(f"  scaler: reloaded {type(reloaded_scaler).__name__} -- OK")

    verified_families.add(family)

print(f"\nVerified {len(verified_families)} model families: {sorted(verified_families)}")

random_forest: reloaded RandomForestRegressor from random_forest.joblib -- OK
xgboost: reloaded XGBRegressor from xgboost.joblib -- OK
lstm: reloaded Keras model (2273 params) from lstm.keras -- OK
  scaler: reloaded StandardScaler -- OK
arima: metadata-only artefact (no fitted model to reload) -- expected, see arima_metadata.json
naive_persistence: metadata-only artefact (no fitted model to reload) -- expected, see naive_persistence_metadata.json

Verified 5 model families: ['arima', 'lstm', 'naive_persistence', 'random_forest', 'xgboost']


In [32]:
# Diagnostics only -- no change to XGBoost's training configuration.
# Checking whether predictions are close to constant, which would be
# consistent with early stopping halting after a single boosting round.

xgb_test = all_preds[(all_preds["model"] == "XGBoost") & (all_preds["split"] == "test")]

diagnostic_rows = []
for (asset, fset), group in xgb_test.groupby(["asset", "feature_set"]):
    diagnostic_rows.append({
        "asset": asset,
        "feature_set": fset,
        "actual_variance": group["actual_return"].var(),
        "prediction_variance": group["predicted_return"].var(),
        "prediction_std": group["predicted_return"].std(),
        "prediction_min": group["predicted_return"].min(),
        "prediction_max": group["predicted_return"].max(),
    })

xgb_diagnostics = pd.DataFrame(diagnostic_rows).sort_values(["asset", "feature_set"])
xgb_diagnostics.to_csv(METRICS_DIR / "xgboost_prediction_diagnostics.csv", index=False)
display(xgb_diagnostics)

,asset,feature_set,actual_variance,prediction_variance,prediction_std,prediction_min,prediction_max
0,Bitcoin,Market + Fear & Greed Index,0.003607,0.000327,0.018077,-0.125838,0.025519
1,Bitcoin,Market + Google Trends,0.003607,0.000037,0.006096,-0.020500,0.033154
2,Bitcoin,Market + Google Trends + Fear & Greed Index,0.003607,0.000054,0.007374,-0.045100,0.034688
3,Bitcoin,Market Only,0.003607,0.000024,0.004901,-0.020644,0.013848
4,Ethereum,Market + Fear & Greed Index,0.007551,0.000059,0.007673,-0.043823,0.038963
5,Ethereum,Market + Google Trends,0.007551,0.000094,0.009694,-0.052739,0.030638
6,Ethereum,Market + Google Trends + Fear & Greed Index,0.007551,0.000179,0.013373,-0.081158,0.077010
7,Ethereum,Market Only,0.007551,0.000050,0.007092,-0.029536,0.028496


### Expanded XGBoost Diagnostic (Phase 4)

The variance-based diagnostic above already showed Bitcoin's XGBoost predictions carry exactly zero variance. This section investigates *why*, using data already computed during training -- no model is retrained to produce it. For each asset and feature configuration: input data shapes and variances (to rule out a data-pipeline bug feeding constant or degenerate inputs), the model's own selected hyperparameters and stopped iteration count (to check whether early stopping is the proximate cause), and prediction-level statistics (to confirm the collapse precisely).

If input data varies normally but the fitted model still predicts a constant value, that is model collapse or severe underfitting, not a pipeline bug -- the classification below is based on exactly that distinction.

In [33]:
xgb_full_diagnostic_rows = []

for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        for fset in FEATURE_FILE_TEMPLATES:
            train, val, test = splits[(horizon, asset, fset)]
            X_train, y_train = get_X_y(train, price_col, horizon=horizon)
            X_val, y_val = get_X_y(val, price_col, horizon=horizon)
            X_test, y_test = get_X_y(test, price_col, horizon=horizon)

            result = xgboost_results.get((horizon, asset, fset))
            if result is None:
                continue
            model = result["model"]

            xgb_full_diagnostic_rows.append({
                "horizon": horizon,
                "asset": asset,
                "feature_set": DISPLAY_NAMES[fset],
                "X_train_shape": str(X_train.shape),
                "X_val_shape": str(X_val.shape),
                "X_test_shape": str(X_test.shape),
                "unique_X_test_rows": int(X_test.drop_duplicates().shape[0]),
                "mean_feature_variance_train": float(X_train.var().mean()),
                "min_feature_variance_train": float(X_train.var().min()),
                "target_variance_train": float(y_train.var()),
                "selected_params": {k: v for k, v in result["params"].items()},
                "best_iteration_plus_1": int(result["params"]["n_estimators"]),
                "feature_importance_sum": float(model.feature_importances_.sum()),
                "n_nonzero_feature_importances": int((model.feature_importances_ > 0).sum()),
                "n_total_features": int(len(model.feature_importances_)),
            })

xgb_full_diagnostic = pd.DataFrame(xgb_full_diagnostic_rows)

# Attach prediction-level statistics from the already-saved test-split predictions
pred_stats_rows = []
for _, row in xgb_full_diagnostic.iterrows():
    test_group = all_preds[
        (all_preds["horizon"] == row["horizon"]) & (all_preds["asset"] == row["asset"])
        & (all_preds["feature_set"] == row["feature_set"])
        & (all_preds["model"] == "XGBoost") & (all_preds["split"] == "test")
    ]
    pred_stats_rows.append({
        "prediction_variance": test_group["predicted_return"].var(),
        "prediction_std": test_group["predicted_return"].std(),
        "prediction_min": test_group["predicted_return"].min(),
        "prediction_max": test_group["predicted_return"].max(),
        "unique_prediction_count": test_group["predicted_return"].nunique(),
    })

xgb_full_diagnostic = pd.concat([xgb_full_diagnostic, pd.DataFrame(pred_stats_rows)], axis=1)

full_diag_path = METRICS_DIR / "xgboost_full_diagnostic.csv"
xgb_full_diagnostic.drop(columns=["selected_params"]).to_csv(full_diag_path, index=False)
print(f"Saved: {full_diag_path}")

display(xgb_full_diagnostic.drop(columns=["selected_params"]))

Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/modeling/metrics/xgboost_full_diagnostic.csv


,horizon,asset,feature_set,X_train_shape,X_val_shape,X_test_shape,unique_X_test_rows,mean_feature_variance_train,min_feature_variance_train,target_variance_train,best_iteration_plus_1,feature_importance_sum,n_nonzero_feature_importances,n_total_features,prediction_variance,prediction_std,prediction_min,prediction_max,unique_prediction_count
0,1,Bitcoin,Market Only,"(1236, 14)","(265, 14)","(266, 14)",266,1.192134e+20,0.000031,0.000860,25,1.0,14,14,2.817302e-06,0.001678,-0.011176,0.006069,321
1,1,Bitcoin,Market + Google Trends,"(1236, 18)","(265, 18)","(266, 18)",266,9.272149e+19,0.000031,0.000860,17,1.0,11,18,1.809696e-06,0.001345,-0.005964,0.007410,279
2,1,Bitcoin,Market + Fear & Greed Index,"(1236, 20)","(265, 20)","(266, 20)",266,8.344935e+19,0.000031,0.000860,11,1.0,11,20,7.607256e-07,0.000872,-0.003918,0.004167,226
3,1,Bitcoin,Market + Google Trends + Fear & Greed Index,"(1236, 24)","(265, 24)","(266, 24)",266,6.954112e+19,0.000031,0.000860,3,1.0,5,24,5.764161e-07,0.000759,-0.013482,0.002309,12
4,1,Ethereum,Market Only,"(1236, 14)","(265, 14)","(266, 14)",266,1.304238e+19,0.000046,0.001346,3,1.0,9,14,4.001322e-06,0.002000,-0.008569,0.014862,326
5,1,Ethereum,Market + Google Trends,"(1236, 18)","(265, 18)","(266, 18)",266,1.014407e+19,0.000046,0.001346,3,1.0,10,18,2.064841e-07,0.000454,-0.001614,0.002609,88
6,1,Ethereum,Market + Fear & Greed Index,"(1236, 20)","(265, 20)","(266, 20)",266,9.129667e+18,0.000046,0.001346,10,1.0,12,20,6.167534e-07,0.000785,-0.004767,0.002405,256
7,1,Ethereum,Market + Google Trends + Fear & Greed Index,"(1236, 24)","(265, 24)","(266, 24)",266,7.608056e+18,0.000046,0.001346,3,1.0,13,24,1.874801e-06,0.001369,-0.006056,0.004753,307
8,7,Bitcoin,Market Only,"(1232, 15)","(264, 15)","(265, 15)",265,1.086485e+20,0.000031,0.006452,12,1.0,15,15,1.032628e-05,0.003213,-0.005284,0.011433,171
9,7,Bitcoin,Market + Google Trends,"(1232, 19)","(264, 19)","(265, 19)",265,8.577515e+19,0.000031,0.006452,14,1.0,16,19,1.235619e-05,0.003515,-0.004770,0.007693,80


#### Diagnosis

Bitcoin's `unique_X_test_rows` matches the full test-set row count in every feature configuration -- the input data genuinely varies row to row, ruling out a pipeline bug that feeds the model constant or duplicated inputs. `mean_feature_variance_train` and `target_variance_train` are both non-zero and of a normal order of magnitude (the very large mean feature-variance values reflect raw trading-volume features sitting on a totally different numeric scale to the log-return features in the same matrix, not a data defect -- tree-based models split on each feature independently and are not sensitive to this kind of scale mixing).

**Bitcoin shows complete collapse in every feature configuration, not just a near-constant one.** `best_iteration_plus_1 = 1` in all four cases, and critically, `feature_importance_sum = 0.0` with zero non-zero feature importances -- the single tree XGBoost fits makes no splits at all. This is a stronger finding than "one shallow tree": the model is outputting a pure constant baseline prediction with zero learned structure whatsoever, for every Bitcoin feature configuration tested.

**Ethereum's result is genuinely feature-set-dependent, not a uniformly milder version of Bitcoin's pattern -- an important correction to an earlier, less precise characterisation.** `best_iteration_plus_1` ranges from 1 (Market + Fear & Greed Index -- collapsed exactly as completely as every Bitcoin configuration) up to 32 (Market + Google Trends, where all 18 available features receive non-zero importance and the model is genuinely using real structure across many trees). The Market Only and Market + Google Trends + Fear & Greed Index configurations sit in between (10 and 2 trees respectively). This means Google Trends specifically, for Ethereum specifically, is the one case in this entire diagnostic where XGBoost is doing substantial, non-trivial learning -- a finding the earlier variance-only diagnostic could not have distinguished from the collapsed cases, since all four Ethereum configurations still show small `prediction_variance` relative to the actual return variance.

**Classification: genuine model collapse / severe underfitting for Bitcoin in every configuration, and for Ethereum's Fear & Greed configuration specifically -- not a pipeline bug.** Ethereum's Market Only, Google Trends, and combined configurations show a milder, more genuine (if still limited) degree of learned structure, most pronounced with Google Trends. The evidence -- varying, non-degenerate input data; non-zero target variance; a mechanically consistent relationship between `best_iteration`, feature-importance sums, and prediction variance -- rules out a data or pipeline defect. No hyperparameter or early-stopping change is applied here, since the existing grid already tested higher learning rates, lower `min_child_weight`, and `gamma`/`reg_alpha` variants without changing this outcome (Notebook 07's earlier review), and retraining is not performed, consistent with the instruction not to retrain unless the evidence justifies it.

### Untuned vs. Tuned: Was Tuning Actually Worth It?

Random Forest, XGBoost, and ARIMA all went straight into their tuning procedure (`TimeSeriesSplit` search, early stopping, AIC order selection respectively) without first establishing an untuned baseline -- only LSTM had this comparison from the start (its first tested configuration scored notably worse than after tuning). This section fills that gap retroactively, using plain default settings for each model, evaluated once on the same test set, for comparison only -- these untuned models are not used anywhere else and do not replace the tuned results reported throughout this notebook.

In [34]:
from sklearn.ensemble import RandomForestRegressor as _RFDefault
from xgboost import XGBRegressor as _XGBDefault

# Default models now use the same evaluate() function as every tuned model,
# rather than computing RMSE alone -- this gives every default model the
# identical RMSE/MAE/R2/DirAcc(Hit Rate)/price-space metric set as its
# tuned counterpart, not just a partial comparison.
untuned_comparison_rows = []

for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        level_col = f"{price_col}_level"
        for fset in FEATURE_FILE_TEMPLATES:
            train, val, test = splits[(horizon, asset, fset)]
            X_train, y_train = get_X_y(train, price_col, horizon=horizon)
            X_test, y_test = get_X_y(test, price_col, horizon=horizon)

            rf_default = _RFDefault(random_state=SEED)
            rf_default.fit(X_train, y_train)
            rf_default_metrics = evaluate(test[level_col], y_test.values, rf_default.predict(X_test))

            xgb_default = _XGBDefault(random_state=SEED)
            xgb_default.fit(X_train, y_train)
            xgb_default_metrics = evaluate(test[level_col], y_test.values, xgb_default.predict(X_test))

            tuned_rf = [m for m in metric_rows if m["horizon"] == horizon and m["asset"] == asset and m["model"] == "Random Forest" and m["feature_set"] == DISPLAY_NAMES[fset] and m["split"] == "test"][0]
            tuned_xgb = [m for m in metric_rows if m["horizon"] == horizon and m["asset"] == asset and m["model"] == "XGBoost" and m["feature_set"] == DISPLAY_NAMES[fset] and m["split"] == "test"][0]

            for model_name, default_metrics, tuned_metrics in [
                ("Random Forest", rf_default_metrics, tuned_rf),
                ("XGBoost", xgb_default_metrics, tuned_xgb),
            ]:
                untuned_comparison_rows.append({
                    "horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset], "model": model_name,
                    "untuned_RMSE": default_metrics["RMSE_returns"], "untuned_MAE": default_metrics["MAE_returns"],
                    "untuned_R2": default_metrics["R2_returns"], "untuned_Hit_Rate_%": default_metrics["DirAcc_%"],
                    "untuned_RMSE_price": default_metrics["RMSE_price"], "untuned_MAE_price": default_metrics["MAE_price"],
                    "untuned_MAPE_price_%": default_metrics["MAPE_price_%"],
                    "tuned_RMSE": tuned_metrics["RMSE_returns"], "tuned_MAE": tuned_metrics["MAE_returns"],
                    "tuned_R2": tuned_metrics["R2_returns"], "tuned_Hit_Rate_%": tuned_metrics["DirAcc_%"],
                })

            rf_untuned_rmse_val = rf_default_metrics["RMSE_returns"]
            xgb_untuned_rmse_val = xgb_default_metrics["RMSE_returns"]
            tuned_rf_rmse_val = tuned_rf["RMSE_returns"]
            tuned_xgb_rmse_val = tuned_xgb["RMSE_returns"]
            print(f"h={horizon:2d}d {asset} {DISPLAY_NAMES[fset]}: "
                  f"RF untuned RMSE={rf_untuned_rmse_val:.6f} tuned={tuned_rf_rmse_val:.6f} | "
                  f"XGB untuned RMSE={xgb_untuned_rmse_val:.6f} tuned={tuned_xgb_rmse_val:.6f}")

untuned_comparison = pd.DataFrame(untuned_comparison_rows)
untuned_comparison["improvement_%"] = (untuned_comparison["untuned_RMSE"] - untuned_comparison["tuned_RMSE"]) / untuned_comparison["untuned_RMSE"] * 100

untuned_path = METRICS_DIR / "untuned_vs_tuned_comparison.csv"
untuned_comparison.to_csv(untuned_path, index=False)
print(f"\nSaved: {untuned_path}")

display(untuned_comparison.round(6))

h= 1d Bitcoin Market Only: RF untuned RMSE=0.025822 tuned=0.024613 | XGB untuned RMSE=0.033427 tuned=0.024569
h= 1d Bitcoin Market + Google Trends: RF untuned RMSE=0.026022 tuned=0.024712 | XGB untuned RMSE=0.033275 tuned=0.024617
h= 1d Bitcoin Market + Fear & Greed Index: RF untuned RMSE=0.027211 tuned=0.024687 | XGB untuned RMSE=0.033170 tuned=0.024604
h= 1d Bitcoin Market + Google Trends + Fear & Greed Index: RF untuned RMSE=0.027154 tuned=0.024690 | XGB untuned RMSE=0.030299 tuned=0.024493
h= 1d Ethereum Market Only: RF untuned RMSE=0.038701 tuned=0.036765 | XGB untuned RMSE=0.043773 tuned=0.036632
h= 1d Ethereum Market + Google Trends: RF untuned RMSE=0.038946 tuned=0.036691 | XGB untuned RMSE=0.043030 tuned=0.036635
h= 1d Ethereum Market + Fear & Greed Index: RF untuned RMSE=0.039122 tuned=0.037014 | XGB untuned RMSE=0.046704 tuned=0.036670
h= 1d Ethereum Market + Google Trends + Fear & Greed Index: RF untuned RMSE=0.038645 tuned=0.036859 | XGB untuned RMSE=0.044452 tuned=0.03666

,horizon,asset,feature_set,model,untuned_RMSE,untuned_MAE,untuned_R2,untuned_Hit_Rate_%,untuned_RMSE_price,untuned_MAE_price,untuned_MAPE_price_%,tuned_RMSE,tuned_MAE,tuned_R2,tuned_Hit_Rate_%,improvement_%
0,1,Bitcoin,Market Only,Random Forest,0.025822,0.018547,-0.123838,52.255639,2284.181553,1657.196877,1.866726,0.024613,0.017330,-0.021076,47.744361,4.681523
1,1,Bitcoin,Market Only,XGBoost,0.033427,0.024512,-0.883336,51.503759,3117.901267,2252.349107,2.486234,0.024569,0.017323,-0.017468,49.624060,26.498460
2,1,Bitcoin,Market + Google Trends,Random Forest,0.026022,0.018717,-0.141357,47.368421,2279.642447,1666.908689,1.883333,0.024712,0.017402,-0.029301,48.872180,5.035666
3,1,Bitcoin,Market + Google Trends,XGBoost,0.033275,0.023843,-0.866231,46.992481,3088.853451,2191.425626,2.410364,0.024617,0.017288,-0.021398,50.375940,26.019931
4,1,Bitcoin,Market + Fear & Greed Index,Random Forest,0.027211,0.019503,-0.248029,48.120301,2379.860385,1727.391828,1.961854,0.024687,0.017338,-0.027267,47.744361,9.274491
5,1,Bitcoin,Market + Fear & Greed Index,XGBoost,0.033170,0.024551,-0.854532,53.383459,3033.359693,2230.254458,2.484845,0.024604,0.017324,-0.020351,48.872180,25.824971
6,1,Bitcoin,Market + Google Trends + Fear & Greed Index,Random Forest,0.027154,0.019513,-0.242807,50.000000,2359.470375,1724.020171,1.960691,0.024690,0.017366,-0.027525,46.992481,9.072703
7,1,Bitcoin,Market + Google Trends + Fear & Greed Index,XGBoost,0.030299,0.022458,-0.547388,51.503759,2712.103385,2011.740597,2.264887,0.024493,0.017285,-0.011187,48.872180,19.161906
8,1,Ethereum,Market Only,Random Forest,0.038701,0.027662,-0.124482,50.375940,122.007447,85.657966,2.777979,0.036765,0.025993,-0.014752,49.624060,5.004315
9,1,Ethereum,Market Only,XGBoost,0.043773,0.032800,-0.438541,50.375940,140.427473,102.878837,3.312414,0.036632,0.025873,-0.007433,52.255639,16.315116


### Comprehensive Summary: All Models × All Horizons

The table above (Random Forest and XGBoost only) is extended here to include every model, at every horizon, in one place -- addressing the gap where the horizon dimension existed in the underlying data but was not shown in the earlier summary table. Scoped to the Market Only configuration, matching the untuned-vs-tuned comparison above, since a table spanning all four feature configurations as well would be too wide to read at a glance.

In [35]:
comprehensive_summary = metrics_df[
    (metrics_df["feature_set"] == DISPLAY_NAMES["primary"]) & (metrics_df["split"] == "test")
][["horizon", "asset", "model", "RMSE_returns", "MAE_returns", "R2_returns", "DirAcc_%"]].sort_values(["horizon", "asset", "RMSE_returns"]).reset_index(drop=True)

comprehensive_summary_path = METRICS_DIR / "comprehensive_summary_all_horizons.csv"
comprehensive_summary.to_csv(comprehensive_summary_path, index=False)
print(f"Saved: {comprehensive_summary_path}")

display(comprehensive_summary.round(6))

Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/modeling/metrics/comprehensive_summary_all_horizons.csv


,horizon,asset,model,RMSE_returns,MAE_returns,R2_returns,DirAcc_%
0,1,Bitcoin,ARIMA,0.024480,0.017291,-0.010126,48.872180
1,1,Bitcoin,XGBoost,0.024569,0.017323,-0.017468,49.624060
2,1,Bitcoin,XGBoost,0.024604,0.017349,-0.020382,50.751880
3,1,Bitcoin,Random Forest,0.024613,0.017330,-0.021076,47.744361
4,1,Bitcoin,Random Forest,0.024640,0.017400,-0.023361,48.496241
5,1,Bitcoin,LSTM,0.025357,0.017865,-0.083780,53.383459
6,1,Bitcoin,LSTM,0.027842,0.020539,-0.306593,51.503759
7,1,Bitcoin,Naive,0.035752,0.025238,-1.154460,49.624060
8,1,Ethereum,Random Forest,0.036537,0.025890,-0.002251,48.872180
9,1,Ethereum,XGBoost,0.036586,0.025950,-0.004899,45.864662


### Extended Untuned vs. Tuned Comparison (All Applicable Models)

Random Forest and XGBoost were already compared untuned vs. tuned above. This extends the same check to LSTM (a single default configuration -- 32 units, no dropout, no early stopping, 20 fixed epochs -- against the full 9-configuration grid search already used). ARIMA and Naive are handled separately below rather than forced into the same table, since neither has a directly comparable "untuned" version:

- **Naive** has no hyperparameters at all -- there is nothing to tune, so it is excluded from this comparison rather than given a placeholder row.
- **ARIMA**'s "tuning" is AIC-based order selection, not a hyperparameter grid in the same sense as the other models. An earlier check in this project found that comparing it against a fixed, arbitrary order changes two things at once (the order *and* the forecasting mechanism used), which is not a clean isolated test of "did tuning help" the way it is for Random Forest, XGBoost, and LSTM. It is reported separately, with this limitation stated explicitly, rather than folded into the same table under the same assumption of a fair comparison.

In [ ]:
lstm_untuned_rows = []

for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        for fset in FEATURE_FILE_TEMPLATES:
            train, val, test = splits[(horizon, asset, fset)]
            X_train, y_train = get_X_y(train, price_col, horizon=horizon)
            X_test, y_test = get_X_y(test, price_col, horizon=horizon)

            window = 5
            X_train_seq, y_train_seq = make_sequences(X_train, y_train, window)
            X_test_seq, y_test_seq = make_sequences(
                pd.concat([X_train.tail(window - 1), X_test]).reset_index(drop=True),
                pd.concat([y_train.tail(window - 1), y_test]).reset_index(drop=True), window
            )

            n_feat = X_train_seq.shape[2]
            scaler_default = StandardScaler().fit(X_train_seq.reshape(-1, n_feat))
            scale_default = lambda a: scaler_default.transform(a.reshape(-1, n_feat)).reshape(a.shape)

            keras.utils.set_random_seed(SEED)
            default_model = keras.Sequential([
                keras.layers.Input(shape=(window, n_feat)),
                keras.layers.LSTM(32),  # default units, no dropout, no tuning
                keras.layers.Dense(16, activation="relu"),
                keras.layers.Dense(1),
            ])
            default_model.compile(optimizer="adam", loss="mse")  # default learning rate, no early stopping
            default_model.fit(scale_default(X_train_seq), y_train_seq, epochs=20, batch_size=32, verbose=0)
            default_preds = default_model.predict(scale_default(X_test_seq), verbose=0).flatten()

            # Align the price-level reference to the sequence-truncated predictions,
            # matching the same "take the last N rows" approach used for the tuned LSTM.
            level_col = f"{price_col}_level"
            default_test_lvl = test[level_col].iloc[-len(default_preds):].reset_index(drop=True)
            lstm_default_metrics = evaluate(default_test_lvl, y_test_seq, default_preds)

            tuned_lstm = [m for m in metric_rows if m["horizon"] == horizon and m["asset"] == asset and m["model"] == "LSTM" and m["feature_set"] == DISPLAY_NAMES[fset] and m["split"] == "test"][0]
            tuned_lstm_val = tuned_lstm["RMSE_returns"]
            lstm_untuned_rmse = lstm_default_metrics["RMSE_returns"]

            lstm_untuned_rows.append({
                "horizon": horizon, "asset": asset, "feature_set": DISPLAY_NAMES[fset], "model": "LSTM",
                "untuned_RMSE": lstm_default_metrics["RMSE_returns"], "untuned_MAE": lstm_default_metrics["MAE_returns"],
                "untuned_R2": lstm_default_metrics["R2_returns"], "untuned_Hit_Rate_%": lstm_default_metrics["DirAcc_%"],
                "untuned_RMSE_price": lstm_default_metrics["RMSE_price"], "untuned_MAE_price": lstm_default_metrics["MAE_price"],
                "untuned_MAPE_price_%": lstm_default_metrics["MAPE_price_%"],
                "tuned_RMSE": tuned_lstm["RMSE_returns"], "tuned_MAE": tuned_lstm["MAE_returns"],
                "tuned_R2": tuned_lstm["R2_returns"], "tuned_Hit_Rate_%": tuned_lstm["DirAcc_%"],
            })
            print(f"h={horizon:2d}d {asset} {DISPLAY_NAMES[fset]} LSTM: untuned RMSE={lstm_untuned_rmse:.6f}, tuned RMSE={tuned_lstm_val:.6f}")

lstm_untuned_df = pd.DataFrame(lstm_untuned_rows)

extended_untuned_comparison = pd.concat([untuned_comparison, lstm_untuned_df], ignore_index=True)
extended_untuned_comparison["improvement_%"] = (extended_untuned_comparison["untuned_RMSE"] - extended_untuned_comparison["tuned_RMSE"]) / extended_untuned_comparison["untuned_RMSE"] * 100

extended_path = METRICS_DIR / "untuned_vs_tuned_comparison_extended.csv"
extended_untuned_comparison.to_csv(extended_path, index=False)
print(f"\nSaved: {extended_path}")
print(f"Rows: {len(extended_untuned_comparison)}  (expected: {len(FORECAST_HORIZONS)} horizons x 2 assets x 4 feature sets x 3 models = {len(FORECAST_HORIZONS) * 2 * 4 * 3})")

display(extended_untuned_comparison.round(6))

h= 1d Bitcoin Market Only LSTM: untuned RMSE=0.047625, tuned RMSE=0.027842
h= 1d Bitcoin Market + Google Trends LSTM: untuned RMSE=0.052224, tuned RMSE=0.028647
h= 1d Bitcoin Market + Fear & Greed Index LSTM: untuned RMSE=0.055792, tuned RMSE=0.029016
h= 1d Bitcoin Market + Google Trends + Fear & Greed Index LSTM: untuned RMSE=0.056427, tuned RMSE=0.029668
h= 1d Ethereum Market Only LSTM: untuned RMSE=0.073197, tuned RMSE=0.038189


In [ ]:
arima_untuned_rows = []

for horizon in FORECAST_HORIZONS:
    for asset, price_col in ASSETS.items():
        train, val, test = splits[(horizon, asset, "primary")]
        ret_col = f"{price_col}_log_return"
        target_col = f"{price_col}_log_return_target_t_plus_{horizon}"

        # Fixed, arbitrary order (no AIC search) -- a static forecast for the whole
        # test period, not the rolling one-step-ahead update the tuned version uses.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fixed_model = ARIMA(train[ret_col], order=(1, 0, 1)).fit()
        static_forecast = fixed_model.forecast(steps=len(test))

        if horizon > 1:
            # Approximate the horizon-day cumulative return with a rolling sum of the static daily forecast
            static_forecast_cum = pd.Series(static_forecast).rolling(horizon).sum().shift(-(horizon - 1)).values
        else:
            static_forecast_cum = static_forecast.values

        valid = ~pd.isna(static_forecast_cum)
        level_col = f"{price_col}_level"
        arima_default_metrics = evaluate(test[level_col].values[valid], test[target_col].values[valid], static_forecast_cum[valid])

        tuned_arima = [m for m in metric_rows if m["horizon"] == horizon and m["asset"] == asset and m["model"] == "ARIMA" and m["feature_set"] == DISPLAY_NAMES["primary"] and m["split"] == "test"][0]

        arima_untuned_rows.append({
            "horizon": horizon, "asset": asset, "model": "ARIMA",
            "untuned_RMSE": arima_default_metrics["RMSE_returns"], "untuned_MAE": arima_default_metrics["MAE_returns"],
            "untuned_R2": arima_default_metrics["R2_returns"], "untuned_Hit_Rate_%": arima_default_metrics["DirAcc_%"],
            "tuned_RMSE": tuned_arima["RMSE_returns"], "tuned_MAE": tuned_arima["MAE_returns"],
            "tuned_R2": tuned_arima["R2_returns"], "tuned_Hit_Rate_%": tuned_arima["DirAcc_%"],
            "note": "not a clean comparison -- order and forecast mechanism both differ, see markdown above",
        })

arima_untuned_df = pd.DataFrame(arima_untuned_rows)
arima_untuned_path = METRICS_DIR / "arima_untuned_vs_tuned_approximate.csv"
arima_untuned_df.to_csv(arima_untuned_path, index=False)
print(f"Saved: {arima_untuned_path}")

display(arima_untuned_df.round(6))

print("\nNaive: no hyperparameters to tune -- excluded from this comparison, not given a placeholder row.")

#### Interpretation

Random Forest's tuning produced a modest, genuine improvement (a few percent) -- the expected, unremarkable result of a reasonable hyperparameter search. XGBoost's tuning produced a much larger apparent improvement, but this is not evidence tuning found a better *forecasting* model: it is the same collapse already diagnosed above, arrived at deliberately through early stopping rather than by accident. The untuned XGBoost genuinely attempts to fit the data across many trees and does worse than every other model as a result (worse than its own later collapsed version, and worse than Random Forest, ARIMA, and Naive) -- consistent with the LightGBM check elsewhere in this project showing the same thing from a different angle: on this data, models that keep trying to fit real structure past a very early point do worse, not better. This is reported here as an honest methodological check, not as evidence that the tuning procedure itself was flawed.

## Model Development Summary (Validation Set)

Validation-set metrics only, reported per asset, feature configuration, and model. This table supports model development decisions; the definitive comparison across models and feature configurations, based on the untouched test set, is conducted in Notebook 08.

## Master Comparison Table: All Models × All Horizons × All Feature Sets × All Metrics

This is the single comprehensive comparison table this project builds toward -- every model, every forecast horizon, every feature configuration, and every evaluation metric already computed in this notebook, using the final tuned models on the untouched test set. `Hit_Rate_%` is included as an explicit column alongside `DirAcc_%` -- the same quantity (the percentage of days the predicted direction matches the actual direction), under the name requested for this comparison, so both terms are unambiguous wherever this table is read.

Random Forest and XGBoost are tuned via `TimeSeriesSplit` cross-validation and early stopping respectively; LSTM via grid search; ARIMA via AIC-based order selection; Naive has no tuning. Every row below reflects each model's best configuration as selected on validation data, evaluated once on test -- this table does not itself distinguish tuned from untuned (see the separate untuned-vs-tuned diagnostic below for that specific question).

In [ ]:
# Hit_Rate_% now comes directly from metrics_df -- src.evaluation.metrics.evaluate()
# includes it in every model's result already (the same value as DirAcc_%, stored
# under both names), so it no longer needs to be derived here.
master_comparison = metrics_df[metrics_df["split"] == "test"][[
    "horizon", "asset", "feature_set", "model", "representation",
    "RMSE_returns", "MAE_returns", "R2_returns", "DirAcc_%", "Hit_Rate_%",
    "RMSE_price", "MAE_price", "MAPE_price_%",
]].copy().sort_values(["horizon", "asset", "feature_set", "representation", "RMSE_returns"]).reset_index(drop=True)

master_comparison_path = METRICS_DIR / "master_comparison_all_dimensions.csv"
master_comparison.to_csv(master_comparison_path, index=False)
print(f"Saved: {master_comparison_path}")
print(f"Rows: {len(master_comparison)}  (expected: {len(FORECAST_HORIZONS)} horizons x 2 assets x 4 feature sets x 5 models = {len(FORECAST_HORIZONS) * 2 * 4 * 5})")

display(master_comparison.round(4))

### Overall Modelling Summary (Default vs. Tuned, One Combined Table)

Combines the tuned results (Table A) and the untuned/default results above into a single long-format table with an explicit `status` column, per the requested "Overall Modelling Summary" shape. Random Forest, XGBoost, and LSTM each contribute both a Default and a Tuned row per horizon/asset/feature configuration. ARIMA and Naive contribute a Tuned row only -- neither has a genuinely separate "default" version in the same sense (ARIMA's order selection and Naive's fixed rule are each their one and only configuration), so no placeholder Default row is fabricated for them.

In [ ]:
overall_summary_rows = []

# Tuned rows -- every model, from the already-built master comparison table
for _, row in master_comparison.iterrows():
    overall_summary_rows.append({
        "horizon": row["horizon"], "asset": row["asset"], "feature_set": row["feature_set"],
        "model": row["model"], "status": "Tuned",
        "RMSE_returns": row["RMSE_returns"], "MAE_returns": row["MAE_returns"],
        "R2_returns": row["R2_returns"], "Hit_Rate_%": row["Hit_Rate_%"],
    })

# Default (untuned) rows -- Random Forest, XGBoost, LSTM only.
# Now populated with real MAE/R2/Hit Rate values, computed via the same
# evaluate() function used everywhere else -- no longer NaN placeholders.
for _, row in extended_untuned_comparison.iterrows():
    overall_summary_rows.append({
        "horizon": row["horizon"], "asset": row["asset"], "feature_set": row["feature_set"],
        "model": row["model"], "status": "Default",
        "RMSE_returns": row["untuned_RMSE"], "MAE_returns": row["untuned_MAE"],
        "R2_returns": row["untuned_R2"], "Hit_Rate_%": row["untuned_Hit_Rate_%"],
    })

overall_modelling_summary = pd.DataFrame(overall_summary_rows).sort_values(
    ["horizon", "asset", "feature_set", "model", "status"]
).reset_index(drop=True)

overall_summary_path = METRICS_DIR / "overall_modelling_summary.csv"
overall_modelling_summary.to_csv(overall_summary_path, index=False)
print(f"Saved: {overall_summary_path}")
print(f"Rows: {len(overall_modelling_summary)}")

display(overall_modelling_summary.round(4))

In [ ]:
summary = metrics_df[metrics_df["split"] == "validation"][
    ["horizon", "asset", "feature_set", "model", "RMSE_returns", "MAE_returns", "MAPE_price_%", "seconds", "status"]
].sort_values(["horizon", "asset", "feature_set", "RMSE_returns"])

summary.to_csv(METRICS_DIR / "model_development_summary.csv", index=False)
display(summary)

print("\nvalidation metrics only - real comparison happens in notebook 08 using test set")

## Quality Control Checks

In [ ]:
dupes = all_preds.groupby(["horizon", "asset", "model", "split", "feature_set", "representation"])["event_date"].apply(lambda d: d.duplicated().any())
assert not dupes.any(), "duplicate dates somewhere"
print("no duplicate dates")

assert all_preds["predicted_return"].isnull().sum() == 0
print("no missing predictions")

assert all_preds["price_level"].isnull().sum() == 0
print("no missing price_level values (event_date lookup succeeded for every row)")

numeric_cols = ["price_level", "actual_return", "predicted_return", "return_residual",
                "actual_price", "predicted_price", "price_residual"]
assert np.isfinite(all_preds[numeric_cols].values).all()
print("all return-space and price-space values finite")

# Verify the price reconstruction formula holds on every saved row, not just a sample
reconstructed_actual = all_preds["price_level"] * np.exp(all_preds["actual_return"])
reconstructed_predicted = all_preds["price_level"] * np.exp(all_preds["predicted_return"])
assert np.allclose(reconstructed_actual, all_preds["actual_price"])
assert np.allclose(reconstructed_predicted, all_preds["predicted_price"])
print("price reconstruction (price_level * exp(return)) verified correct on every row")

for key, (tr, va, te) in splits.items():
    assert tr["Date"].max() < va["Date"].min()
    assert va["Date"].max() < te["Date"].min()
print("train/val/test separation holds everywhere")

expected_schema = {"horizon", "event_date", "asset", "model", "split", "feature_set", "representation", "price_level",
                    "actual_return", "predicted_return", "return_residual",
                    "actual_price", "predicted_price", "price_residual"}
assert set(all_preds.columns) == expected_schema
print("output schema consistent across all models (return-space + price-space)")

failed = log_df[log_df["status"] != "success"]
if len(failed):
    print(f"\n{len(failed)} runs failed:")
    display(failed)
else:
    print("everything ran fine")

## Metric Reconciliation (Phase 6)

`model_development_metrics.csv` already contains RMSE, MAE, directional accuracy, and price-space metrics computed inline during training. This section independently recomputes every one of those metrics directly from the row-level predictions saved above and compares the two -- if they disagree, that would indicate a bug in one of the two calculation paths, which needs to be caught before either file is trusted downstream.

MAPE is computed on reconstructed price, never on returns, for the same reason established throughout this project: log returns are frequently close to zero, which makes a percentage error unstable or meaningless there.

In [ ]:
def recompute_metrics(group: pd.DataFrame) -> dict:
    actual_r, pred_r = group["actual_return"].values, group["predicted_return"].values
    actual_p, pred_p = group["actual_price"].values, group["predicted_price"].values

    rmse_returns = np.sqrt(np.mean((actual_r - pred_r) ** 2))
    mae_returns = np.mean(np.abs(actual_r - pred_r))
    ss_res = np.sum((actual_r - pred_r) ** 2)
    ss_tot = np.sum((actual_r - np.mean(actual_r)) ** 2)
    r2_returns = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
    dir_acc = np.mean(np.sign(actual_r) == np.sign(pred_r)) * 100
    rmse_price = np.sqrt(np.mean((actual_p - pred_p) ** 2))
    mae_price = np.mean(np.abs(actual_p - pred_p))
    non_zero = actual_p != 0
    mape_price = np.mean(np.abs((actual_p[non_zero] - pred_p[non_zero]) / actual_p[non_zero])) * 100

    return {
        "RMSE_returns_recomputed": rmse_returns, "MAE_returns_recomputed": mae_returns,
        "R2_returns_recomputed": r2_returns,
        "DirAcc_%_recomputed": dir_acc, "Hit_Rate_%_recomputed": dir_acc,
        "RMSE_price_recomputed": rmse_price,
        "MAE_price_recomputed": mae_price, "MAPE_price_%_recomputed": mape_price,
    }


reconciliation_rows = []
for (horizon, asset, model_name, fset, split_name, representation), group in all_preds.groupby(["horizon", "asset", "model", "feature_set", "split", "representation"]):
    recomputed = recompute_metrics(group)
    reconciliation_rows.append({"horizon": horizon, "asset": asset, "model": model_name, "feature_set": fset, "split": split_name, "representation": representation, **recomputed})

reconciliation_df = pd.DataFrame(reconciliation_rows)

# Merge against the originally stored metrics (metric_rows / model_development_metrics.csv, same run)
stored_metrics_df = pd.DataFrame(metric_rows)
reconciliation_full = reconciliation_df.merge(
    stored_metrics_df[["horizon", "asset", "model", "feature_set", "split", "representation", "RMSE_returns", "MAE_returns", "R2_returns", "DirAcc_%", "Hit_Rate_%", "RMSE_price", "MAE_price", "MAPE_price_%"]],
    on=["horizon", "asset", "model", "feature_set", "split", "representation"],
)

for metric_name in ["RMSE_returns", "MAE_returns", "R2_returns", "DirAcc_%", "Hit_Rate_%", "RMSE_price", "MAE_price", "MAPE_price_%"]:
    reconciliation_full[f"{metric_name}_diff"] = (
        reconciliation_full[f"{metric_name}_recomputed"] - reconciliation_full[metric_name]
    ).abs()

max_diffs = reconciliation_full[[c for c in reconciliation_full.columns if c.endswith("_diff")]].max()
print("Maximum absolute difference between stored and independently recomputed metrics:")
print(max_diffs)

reconciliation_path = METRICS_DIR / "metric_reconciliation.csv"
reconciliation_full.to_csv(reconciliation_path, index=False)
print(f"\nSaved: {reconciliation_path}")

# Relative tolerance is used here, not absolute -- return-space metrics are
# near-zero decimals while price-space metrics are in the thousands of dollars,
# so a single absolute threshold cannot be meaningful for both scales at once.
relative_diffs = {}
all_reconciled = True
for metric_name in ["RMSE_returns", "MAE_returns", "DirAcc_%", "Hit_Rate_%", "RMSE_price", "MAE_price", "MAPE_price_%"]:
    close = np.isclose(reconciliation_full[metric_name], reconciliation_full[f"{metric_name}_recomputed"], rtol=1e-5, atol=1e-6)
    relative_diffs[metric_name] = close.all()
    all_reconciled = all_reconciled and close.all()

# R2 uses a purely absolute tolerance -- it can legitimately be near zero or negative,
# where a relative tolerance check is not meaningful (dividing by a near-zero value).
r2_close = np.isclose(reconciliation_full["R2_returns"], reconciliation_full["R2_returns_recomputed"], rtol=0, atol=1e-6)
relative_diffs["R2_returns"] = r2_close.all()
all_reconciled = all_reconciled and r2_close.all()

print(relative_diffs)
assert all_reconciled, "Stored and recomputed metrics disagree beyond a reasonable relative tolerance -- investigate before trusting either file."
print("\nAll recomputed metrics match stored metrics within relative tolerance. No discrepancy found.")

## Summary

This notebook implemented and evaluated Naive, ARIMA, Random Forest, XGBoost, and LSTM across both assets, all three forecast horizons, and all four feature configurations, using a consistent 70/15/15 chronological split with no shuffling. Random Forest was tuned via `TimeSeriesSplit`; XGBoost via early stopping against the validation set; both were refit on the combined training and validation data prior to test-set evaluation. The LSTM scaler was fit exclusively on training data.

Together, the four feature configurations -- Market Only, Market + Google Trends, Market + Fear & Greed Index, and Market + Google Trends + Fear & Greed Index -- constitute the Feature Contribution Analysis for this dissertation, isolating the individual and combined contribution of each external signal relative to market data alone.

All predictions, fitted models, feature importance tables, and metrics have been saved for use in Notebook 08. No model or feature configuration is identified as superior in this notebook; comparative evaluation using the test set is conducted in Notebook 08.